# Q-Learning  
### *Model-free RL, off-policy method*

#### Load the Tic-Tac-Toe environment.

In [84]:
from tic_tac_toe_env import TicTacToe
import random

In [85]:
def random_move(game: TicTacToe, letter: str):
    board = game.board
    player = letter
    opponent = 'O' if player == 'X' else 'X'

    move = random.choice(game.available_moves())
    return move

In [57]:
import random

def choose_move(game: TicTacToe, letter: str, epsilon=1.0):
    """epsilon = probability of intentionally choosing a suboptimal move"""
    board = game.board
    player = letter
    opponent = 'O' if player == 'X' else 'X'

    def empty():
        return game.available_moves()

    def lines():
        return [
            [0,1,2], [3,4,5], [6,7,8],
            [0,3,6], [1,4,7], [2,5,8],
            [0,4,8], [2,4,6]
        ]

    def can_win(p):
        for line in lines():
            vals = [board[i] for i in line]
            if vals.count(p) == 2 and vals.count(' ') == 1:
                return line[vals.index(' ')]
        return None

    def creates_fork(p, idx):
        board[idx] = p
        wins = 0
        for line in lines():
            vals = [board[i] for i in line]
            if vals.count(p) == 2 and vals.count(' ') == 1:
                wins += 1
        board[idx] = ' '
        return wins >= 2

    center  = 4
    corners = [0,2,6,8]
    edges   = [1,3,5,7]

    # --- 1. Win ---
    move = can_win(player)
    if move is not None:
        return move

    # --- 2. Block ---
    move = can_win(opponent)
    if move is not None:
        return move

    # --- 3. Create fork ---
    for m in empty():
        if creates_fork(player, m):
            return m

    # --- 4. Block opponent fork ---
    opp_forks = [m for m in empty() if creates_fork(opponent, m)]
    if len(opp_forks) == 1:
        return opp_forks[0]

    if len(opp_forks) > 1:
        for m in empty():
            board[m] = player
            if can_win(player) is not None:
                board[m] = ' '
                return m
            board[m] = ' '

    # --- At this point: we introduce controlled randomness ---
    if random.random() < epsilon:
        # choose *any legal move* except immediate losing ones
        moves = empty()
        random.shuffle(moves)
        return moves[0]

    # --- 5. Center ---
    if center in empty():
        return center

    # --- 6. Opposite corner ---
    opposite = [(0,8),(8,0),(2,6),(6,2)]
    for a,b in opposite:
        if board[a] == opponent and b in empty():
            return b

    # --- 7. Corner ---
    for c in corners:
        if c in empty():
            return c

    # --- 8. Edge ---
    for e in edges:
        if e in empty():
            return e

    return empty()[0]

#### Setup the Q-Learning Agent.

In [ ]:
import random
import pickle

class QLearningAgent:
    def __init__(self, alpha=0.5, gamma=0.9, epsilon=0.1):
        self.q_table = {}     # (state_tuple, action) → Q
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, game):
        """Epsilon-greedy action selection using Q-values."""
        state = tuple(game.board)
        available = game.available_moves()

        # Exploration
        if random.random() < self.epsilon:
            return random.choice(available)

        # Exploitation
        q_values = [self.get_q(state, a) for a in available]
        max_q = max(q_values)
        best_moves = [a for a, q in zip(available, q_values) if q == max_q]
        return random.choice(best_moves)

    def best_action(self, game):
        """Choose best move with no randomness for eval"""
        state = tuple(game.board)
        available = game.available_moves()

        q_values = [self.get_q(state, a) for a in available]
        max_q = max(q_values)
        best_moves = [a for a, q in zip(available, q_values) if q == max_q]
        return random.choice(best_moves)

    def learn(self, state, action, reward, next_state, next_available, done):
        """q-learning update"""
        old_q = self.get_q(state, action)

        if done:
            target = reward
        else:
            future_q = max(self.get_q(next_state, a) for a in next_available)
            target = reward + self.gamma * future_q

        new_q = old_q + self.alpha * (target - old_q)
        self.q_table[(state, action)] = new_q  # Store the updated Q-value


    # save weights
    def save(self, filename="q_table.pkl"):
        with open(filename, "wb") as f:
            pickle.dump(self.q_table, f)

    def load(self, filename="q_table.pkl"):
        with open(filename, "rb") as f:
            self.q_table = pickle.load(f)


#### Train the Q-Learning Agent

In [ ]:
def train(agent_x, agent_o, episodes=200000, n=3, epsilon_decay=0.9999):
    epsilon = 1.0  # Sstattart with full exploration
    for episode in range(episodes):
        game = TicTacToe(n=n)
        state_x = tuple(game.board)
        state_o = state_x
        letter = 'X'  # X always starts in training

        while True:
            if letter == 'X':
                action = agent_x.choose_action(game)
                game.make_move(action, letter)
                next_state = tuple(game.board)
                next_available = game.available_moves()

                # Get reward
                reward = get_reward(game, 'X')
                done = game.current_winner == 'X' or not game.empty_squares()
                agent_x.learn(state_x, action, reward, next_state, next_available, done)

                if done:
                    break
                state_x = next_state
                letter = 'O'
            else:  # now it is O's turn
                action = agent_o.choose_action(game)
                game.make_move(action, letter)
                next_state = tuple(game.board)
                next_available = game.available_moves()

                # Get reward
                reward = get_reward(game, 'O')
                done = game.current_winner == 'O' or not game.empty_squares()
                agent_o.learn(state_o, action, reward, next_state, next_available, done)

                if done:
                    break
                state_o = next_state
                letter = 'X'

        # Decay our epsilon after each episode to reduce exploration
        epsilon *= epsilon_decay
        agent_x.epsilon = epsilon
        agent_o.epsilon = epsilon



In [ ]:
def get_reward(game, letter):
    """the rewards for winning, losing, and non-terminal states."""
    if game.current_winner == letter:
        return 1  # Positive reward for winning
    elif game.current_winner == 'O' and letter == 'X':
        return -1  # Negative reward for losing
    elif game.current_winner == 'X' and letter == 'O':
        return -1
    elif not game.empty_squares():
        return 0  # Draw
    else:
        return 0  # Non-terminal state soo neutral reward


In [185]:
#5000000
agentX = QLearningAgent()
agentO = QLearningAgent()

train(agentX, agentO, episodes=2000000)

In [192]:
def play_vs_random(agent, n=3, print_game=False):
    game = TicTacToe(n=n)

    agent_letter = 'O'
    random_letter = 'X'
    turn = 'O'

    # Use deterministic best moves
    agent.epsilon = 0

    while True:
        if print_game:
            print(game)
            print()

        if turn == agent_letter:
            move = agent.best_action(game)
            game.make_move(move, agent_letter)
        else:
            move = random_move(game, random_letter)
            game.make_move(move, random_letter)

        # Check win
        if game.current_winner:
            if turn == agent_letter:
                return 1      # Q-agent wins
            else:
                return -1     # random agent wins

        # Check draw
        if not game.empty_squares():
            return 0

        # Swap turn
        turn = random_letter if turn == agent_letter else agent_letter


In [187]:
def evaluate(agent, games=1000, n=3):
    wins = draws = losses = 0

    for _ in range(games):
        result = play_vs_random(agent, n=n, print_game=False)

        if result == 1:
            wins += 1
        elif result == 0:
            draws += 1
        else:
            losses += 1

    print(f"Games: {games}")
    print(f"Wins:  {wins/games * 100} %")
    print(f"Draws: {draws/games * 100} %")
    print(f"Losses:{losses/games * 100}%")


In [194]:
# agent = QLearningAgent()
# agent.load("q_table.pkl")   # load trained weights
# these are against random agent, win rate is m
evaluate(agentO, games=15000, n=3)


Games: 15000
Wins:  97.09333333333333 %
Draws: 1.2133333333333334 %
Losses:1.6933333333333336%
